# 07 - Deep Dive : les angles qui parlent aux fans de foot

70 000 tirs. 2 800 matchs. 11 ligues.
On creuse plus profond avec des questions que les fans se posent vraiment.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

plt.rcParams.update({"font.family": "sans-serif", "font.size": 13,
    "axes.titlesize": 16, "axes.titleweight": "bold", "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "figure.dpi": 150,
    "axes.spines.top": False, "axes.spines.right": False})

BG = "#FAFAFA"
RED = "#E63946"
BLUE = "#457B9D"
GREEN = "#2A9D8F"
ORANGE = "#F4A261"
DARK = "#1D3557"
GRAY = "#ADB5BD"
GOLD = "#E9C46A"
PURPLE = "#6A4C93"
print("Setup OK")

In [ ]:
# Charger le dataset complet
shots = pd.read_parquet("../data/processed/shots_clean_all.parquet")
timeline = pd.read_parquet("../data/processed/timelines_all.parquet")

from src.features import UnderperformanceFeatures
feat = UnderperformanceFeatures()
tl = feat.filter_complete_windows(timeline, window=10)

# Mapper league
match_league = shots[["match_id", "league"]].drop_duplicates()
tl = tl.merge(match_league, on="match_id", how="left")

n_shots = len(shots)
n_matchs = shots["match_id"].nunique()
n_leagues = shots["league"].nunique()
print(f"Dataset : {n_shots} tirs, {n_matchs} matchs, {n_leagues} ligues")
print(f"Timeline filtree : {len(tl)} observations")

---
## 1. Est-ce que la punition arrive plus tard ?

Peut-etre que 10 minutes c est trop court.
Testons 5, 10, 15, 20 et 30 minutes.

In [ ]:
from src.analysis import WindowAnalysis
wa = WindowAnalysis()

# Construire les fenetres 15 et 20 min manuellement
# On a deja 5 et 10 dans le dataset. Pour 15/20 on recalcule.
windows_to_test = [5, 10, 15]
window_results = []

for w in windows_to_test:
    col_c = f"future_conceded_{w}min"
    if col_c not in timeline.columns:
        print(f"Fenetre {w}min pas disponible, skip")
        continue
    
    tl_w = feat.filter_complete_windows(timeline, window=w)
    tl_w = tl_w.merge(match_league, on="match_id", how="left")
    
    for thresh in [0.3, 0.5, 0.8]:
        res = wa.event_study(tl_w, threshold=thresh, window=w)
        if res is None:
            continue
        diff = res["concede"]["difference"]
        pval = res["concede"]["p_value"]
        n_t = res["concede"]["n_treated"]
        window_results.append({"window": w, "threshold": thresh,
                               "diff": diff, "p_value": pval, "n_treated": n_t})

wr_df = pd.DataFrame(window_results)
print(wr_df.to_string(index=False))

In [ ]:
# Visualisation : effet par fenetre temporelle
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

colors_t = {0.3: BLUE, 0.5: RED, 0.8: PURPLE}
for thresh in [0.3, 0.5, 0.8]:
    sub = wr_df[wr_df["threshold"] == thresh]
    if len(sub) == 0:
        continue
    label = "Seuil " + str(thresh) + " xG"
    style = "-o" if thresh == 0.5 else "--s"
    ax.plot(sub["window"], sub["diff"] * 100, style, color=colors_t[thresh],
            linewidth=2.5, markersize=8, label=label)

ax.axhline(0, color=DARK, linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Fenetre temporelle (minutes)")
ax.set_ylabel("Delta P(conceder) en %")
ax.set_title("La punition arrive-t-elle plus tard ?", pad=15)
ax.legend(fontsize=12)

ax.annotate("Sous zero = les equipes qui gaspillent
concedent MOINS, pas plus",
            xy=(0.98, 0.05), xycoords="axes fraction", ha="right", va="bottom",
            fontsize=11, color=GRAY, style="italic")

plt.tight_layout()
plt.savefig("../outputs/figures/pub_08_windows.png", dpi=200, bbox_inches="tight", facecolor=BG)
plt.show()

---
## 2. La malediction de l occasion ratee

Quand une equipe rate un tir a xG > 0.5 ("une occasion en or"),
que se passe-t-il dans les 10 minutes suivantes ?

In [ ]:
# Identifier les "grosses occasions ratees"
big_chances = shots[(shots["xg"] >= 0.4) & (shots["is_goal"] == 0)].copy()
big_chances_scored = shots[(shots["xg"] >= 0.4) & (shots["is_goal"] == 1)].copy()

n_missed = len(big_chances)
n_scored = len(big_chances_scored)
total_big = n_missed + n_scored
miss_rate = n_missed / total_big * 100
print(f"Grosses occasions (xG >= 0.4) : {total_big}")
print(f"  Ratees : {n_missed} ({miss_rate:.1f}%)")
print(f"  Marquees : {n_scored} ({100-miss_rate:.1f}%)")

In [ ]:
# Pour chaque grosse occasion ratee, regarder ce qui se passe apres
# On utilise la timeline : trouver la minute du rate et regarder future_conceded

post_miss = []
post_score = []

for _, shot in big_chances.iterrows():
    mid = shot["match_id"]
    team = shot["team"]
    minute = int(shot["game_minute"])
    
    tl_row = timeline[(timeline["match_id"] == mid) & 
                       (timeline["team"] == team) & 
                       (timeline["minute"] == minute)]
    if len(tl_row) > 0:
        row = tl_row.iloc[0]
        if row.get("future_window_complete_10min", 0) == 1:
            post_miss.append({
                "conceded_10": row["future_conceded_10min"],
                "xga_10": row["future_xga_10min"],
                "opp_shots_10": row["future_opp_shots_10min"],
                "xg_of_shot": shot["xg"],
                "minute": minute,
            })

for _, shot in big_chances_scored.iterrows():
    mid = shot["match_id"]
    team = shot["team"]
    minute = int(shot["game_minute"])
    
    tl_row = timeline[(timeline["match_id"] == mid) & 
                       (timeline["team"] == team) & 
                       (timeline["minute"] == minute)]
    if len(tl_row) > 0:
        row = tl_row.iloc[0]
        if row.get("future_window_complete_10min", 0) == 1:
            post_score.append({
                "conceded_10": row["future_conceded_10min"],
                "xga_10": row["future_xga_10min"],
                "opp_shots_10": row["future_opp_shots_10min"],
                "xg_of_shot": shot["xg"],
                "minute": minute,
            })

miss_df = pd.DataFrame(post_miss)
score_df = pd.DataFrame(post_score)

p_miss = miss_df["conceded_10"].mean() * 100
p_score = score_df["conceded_10"].mean() * 100
n_miss = len(miss_df)
n_score = len(score_df)

print(f"Apres un RATE (xG >= 0.4) : P(conceder 10min) = {p_miss:.1f}%  (n={n_miss})")
print(f"Apres un BUT  (xG >= 0.4) : P(conceder 10min) = {p_score:.1f}%  (n={n_score})")
print(f"Difference : {p_miss - p_score:+.1f} points de %")

In [ ]:
# Visualisation : apres un rate vs apres un but
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

# Panel A : P(conceder)
ax = axes[0]
ax.set_facecolor(BG)
bars = ax.bar([0, 1], [p_miss, p_score], color=[RED, GREEN],
              edgecolor="white", linewidth=2, width=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Occasion ratee", "But marque"], fontsize=13)
ax.set_ylabel("P(conceder dans 10 min) %")
ax.set_title("Apres une grosse occasion (xG >= 0.4)", pad=15)
ax.text(0, p_miss + 0.5, str(round(p_miss, 1)) + "%", ha="center", fontsize=14, fontweight="bold", color=RED)
ax.text(1, p_score + 0.5, str(round(p_score, 1)) + "%", ha="center", fontsize=14, fontweight="bold", color=GREEN)
ax.text(0, -2, "n=" + str(n_miss), ha="center", fontsize=10, color=GRAY)
ax.text(1, -2, "n=" + str(n_score), ha="center", fontsize=10, color=GRAY)

# Panel B : xGA moyen
ax = axes[1]
ax.set_facecolor(BG)
xga_miss = miss_df["xga_10"].mean()
xga_score = score_df["xga_10"].mean()
ax.bar([0, 1], [xga_miss, xga_score], color=[RED, GREEN],
       edgecolor="white", linewidth=2, width=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Occasion ratee", "But marque"], fontsize=13)
ax.set_ylabel("xG Against moyen (10 min)")
ax.set_title("Danger subi apres l occasion", pad=15)
ax.text(0, xga_miss + 0.005, str(round(xga_miss, 3)), ha="center", fontsize=14, fontweight="bold", color=RED)
ax.text(1, xga_score + 0.005, str(round(xga_score, 3)), ha="center", fontsize=14, fontweight="bold", color=GREEN)

fig.suptitle("La malediction de l occasion ratee existe-t-elle ?",
             fontsize=17, fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig("../outputs/figures/pub_09_missed_chance.png", dpi=200, bbox_inches="tight", facecolor=BG)
plt.show()

---
## 3. Premiere mi-temps vs deuxieme mi-temps

La fatigue et la pression changent-elles la donne ?
Si tu gaspilles en 1ere mi-temps, tu le paies en 2eme ?

In [ ]:
# Separer 1ere et 2eme mi-temps
tl_1h = tl[tl["minute"] <= 45].copy()
tl_2h = tl[tl["minute"] > 45].copy()

from src.analysis import WindowAnalysis
wa = WindowAnalysis()

# Test par mi-temps
halves = [("1ere mi-temps", tl_1h), ("2eme mi-temps", tl_2h)]
half_results = []

for name, data in halves:
    for thresh in [0.3, 0.5]:
        res = wa.event_study(data, threshold=thresh, window=10)
        if res is None:
            continue
        d = res["concede"]["difference"]
        p = res["concede"]["p_value"]
        nt = res["concede"]["n_treated"]
        nc = res["concede"]["n_control"]
        half_results.append({"half": name, "threshold": thresh, "diff": d,
                            "p_value": p, "n_treated": nt, "n_control": nc})

hr_df = pd.DataFrame(half_results)
print(hr_df.to_string(index=False))

In [ ]:
# Visualisation mi-temps
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Seuil 0.5 seulement pour la clarte
h05 = hr_df[hr_df["threshold"] == 0.5]
if len(h05) >= 2:
    x = [0, 1]
    vals = h05["diff"].values * 100
    colors = [BLUE, ORANGE]
    bars = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=2, width=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(["1ere mi-temps", "2eme mi-temps"], fontsize=13)
    ax.axhline(0, color=DARK, linestyle="--", linewidth=1)
    ax.set_ylabel("Delta P(conceder) en %")
    ax.set_title("L effet de l underperformance par mi-temps (seuil 0.5)", pad=15)
    
    for i, v in enumerate(vals):
        p = h05.iloc[i]["p_value"]
        sig = " *" if p < 0.05 else ""
        ax.text(i, v + (0.2 if v >= 0 else -0.4), str(round(v, 1)) + "%" + sig,
                ha="center", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/figures/pub_10_by_half.png", dpi=200, bbox_inches="tight", facecolor=BG)
plt.show()

---
## 4. Le siege infructueux

Quand une equipe tire 5+ fois sans marquer d affile,
est-ce que la prochaine action adverse est plus dangereuse ?

In [ ]:
# Identifier les "sieges" : sequences de 5+ tirs sans but
siege_data = []

for match_id in tl["match_id"].unique():
    match_tl = tl[tl["match_id"] == match_id]
    for team in match_tl["team"].unique():
        team_tl = match_tl[match_tl["team"] == team].sort_values("minute")
        
        for _, row in team_tl.iterrows():
            n_shots = row["shots_since_last_goal"]
            if n_shots >= 5:
                siege_data.append({
                    "match_id": match_id, "team": team,
                    "minute": row["minute"], "shots_without_goal": n_shots,
                    "xg_wasted": row["xg_since_last_goal"],
                    "conceded_10": row["future_conceded_10min"],
                    "xga_10": row["future_xga_10min"],
                    "score_state": row["score_state"],
                })

siege_df = pd.DataFrame(siege_data)
print(f"Minutes en siege (5+ tirs sans but) : {len(siege_df)}")

# Comparer : siege vs pas siege
not_siege = tl[tl["shots_since_last_goal"] < 5]

p_siege = siege_df["conceded_10"].mean() * 100
p_normal = not_siege["future_conceded_10min"].mean() * 100
print(f"
P(conceder 10min) pendant un siege : {p_siege:.1f}%  (n={len(siege_df)})")
print(f"P(conceder 10min) hors siege       : {p_normal:.1f}%  (n={len(not_siege)})")
print(f"Difference : {p_siege - p_normal:+.1f} points de %")

In [ ]:
# Gradient : P(conceder) par nombre de tirs sans but
bins_shots = [0, 1, 2, 3, 4, 5, 7, 10, 99]
labels_shots = ["0", "1", "2", "3", "4", "5-6", "7-9", "10+"]
tl_sg = tl.copy()
tl_sg["shots_bin"] = pd.cut(tl_sg["shots_since_last_goal"], bins=bins_shots, labels=labels_shots, right=False)

sg_stats = tl_sg.groupby("shots_bin", observed=True).agg(
    p_concede=("future_conceded_10min", "mean"),
    n=("future_conceded_10min", "count")
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Gradient de couleur
n_bars = len(sg_stats)
cmap = plt.cm.YlOrRd
colors = [cmap(i / n_bars) for i in range(n_bars)]

ax.bar(range(n_bars), sg_stats["p_concede"] * 100, color=colors,
       edgecolor="white", linewidth=1.5, width=0.7)

baseline = tl["future_conceded_10min"].mean() * 100
ax.axhline(baseline, color=DARK, linestyle="--", linewidth=1.5, alpha=0.5)
ax.text(n_bars - 0.5, baseline + 0.2, "Moyenne", ha="right", fontsize=10, color=DARK)

ax.set_xticks(range(n_bars))
ax.set_xticklabels(sg_stats["shots_bin"], fontsize=11)
ax.set_xlabel("Nombre de tirs sans marquer (d affile)")
ax.set_ylabel("P(conceder dans 10 min) %")
ax.set_title("Le siege infructueux rend-il vulnerable ?", pad=15)

for i, row in sg_stats.iterrows():
    p = row["p_concede"] * 100
    n = int(row["n"])
    ax.text(i, p + 0.3, str(round(p, 1)) + "%", ha="center", fontsize=10, fontweight="bold")
    ax.text(i, -0.8, "n=" + str(n), ha="center", fontsize=8, color=GRAY)

ax.set_ylim(bottom=-1.5)
plt.tight_layout()
plt.savefig("../outputs/figures/pub_11_siege.png", dpi=200, bbox_inches="tight", facecolor=BG)
plt.show()

---
## 5. Le classement du gaspillage

Quelles equipes gaspillent le plus ? Et est-ce que ca les punit ?

In [ ]:
# xG - Buts par equipe (total saison)
team_stats = shots.groupby(["team", "league"]).agg(
    total_xg=("xg", "sum"),
    total_goals=("is_goal", "sum"),
    n_shots=("xg", "count"),
    n_matches=("match_id", "nunique")
).reset_index()
team_stats["underperf"] = team_stats["total_xg"] - team_stats["total_goals"]
team_stats["underperf_per_match"] = team_stats["underperf"] / team_stats["n_matches"]
team_stats["conversion"] = team_stats["total_goals"] / team_stats["n_shots"] * 100

# Filtrer les equipes avec assez de matchs
team_stats = team_stats[team_stats["n_matches"] >= 10]

# Top gaspilleurs
top_waste = team_stats.sort_values("underperf", ascending=False).head(15)
print("TOP 15 GASPILLEURS :")
for _, row in top_waste.iterrows():
    t = row["team"]
    l = row["league"]
    u = row["underperf"]
    xg = row["total_xg"]
    g = int(row["total_goals"])
    nm = int(row["n_matches"])
    print(f"  {t:<25s} ({l:<15s}) {u:>+6.1f} xG  ({xg:.0f} xG, {g} buts, {nm} matchs)")

In [ ]:
# Visualisation : scatter xG per match vs concession per match
# Calculer les concessions par equipe
team_concede = []
for team in team_stats["team"].unique():
    team_tl = tl[tl["team"] == team]
    if len(team_tl) < 100:
        continue
    p_c = team_tl["future_conceded_10min"].mean()
    team_concede.append({"team": team, "p_concede": p_c})

tc_df = pd.DataFrame(team_concede)
merged = team_stats.merge(tc_df, on="team")

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

scatter = ax.scatter(merged["underperf_per_match"], merged["p_concede"] * 100,
                     s=merged["n_matches"] * 3, alpha=0.6, c=merged["underperf_per_match"],
                     cmap="RdYlGn_r", edgecolors="white", linewidths=0.5)

# Annoter les extremes
for _, row in merged.nlargest(5, "underperf_per_match").iterrows():
    ax.annotate(row["team"], xy=(row["underperf_per_match"], row["p_concede"]*100),
                fontsize=9, ha="left", va="bottom", color=DARK)
for _, row in merged.nsmallest(3, "underperf_per_match").iterrows():
    ax.annotate(row["team"], xy=(row["underperf_per_match"], row["p_concede"]*100),
                fontsize=9, ha="left", va="bottom", color=DARK)

# Tendance
z = np.polyfit(merged["underperf_per_match"], merged["p_concede"]*100, 1)
x_line = np.linspace(merged["underperf_per_match"].min(), merged["underperf_per_match"].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), "--", color=RED, linewidth=2, alpha=0.7)

# Correlation
r, p = stats.pearsonr(merged["underperf_per_match"], merged["p_concede"])
ax.text(0.02, 0.95, f"r = {r:.3f}, p = {p:.3f}",
        transform=ax.transAxes, fontsize=12, va="top",
        bbox=dict(facecolor="white", edgecolor=GRAY, alpha=0.9))

ax.set_xlabel("Underperformance par match (xG - Buts)")
ax.set_ylabel("P(conceder dans 10 min) %")
ax.set_title("Les equipes qui gaspillent concedent-elles plus ?", pad=15)
plt.colorbar(scatter, label="Underperf/match", ax=ax)
plt.tight_layout()
plt.savefig("../outputs/figures/pub_12_team_scatter.png", dpi=200, bbox_inches="tight", facecolor=BG)
plt.show()

---
## 6. Le penalty rate : la pire des frustrations

Un penalty rate, c est le summum du gaspillage.
Est-ce que ca casse psychologiquement une equipe ?

In [ ]:
# Penalties rates
pens = shots[shots["is_penalty"] == 1].copy()
pens_missed = pens[pens["is_goal"] == 0]
pens_scored = pens[pens["is_goal"] == 1]

n_pen = len(pens)
n_pen_missed = len(pens_missed)
n_pen_scored = len(pens_scored)
print(f"Penalties : {n_pen} total")
print(f"  Rates : {n_pen_missed} ({n_pen_missed/n_pen*100:.1f}%)")
print(f"  Marques : {n_pen_scored} ({n_pen_scored/n_pen*100:.1f}%)")

# Outcomes apres
post_pen_miss = []
post_pen_score = []

for _, shot in pens_missed.iterrows():
    mid = shot["match_id"]
    team = shot["team"]
    minute = int(shot["game_minute"])
    tl_row = timeline[(timeline["match_id"] == mid) & (timeline["team"] == team) & (timeline["minute"] == minute)]
    if len(tl_row) > 0 and tl_row.iloc[0].get("future_window_complete_10min", 0) == 1:
        post_pen_miss.append({"conceded": tl_row.iloc[0]["future_conceded_10min"],
                              "xga": tl_row.iloc[0]["future_xga_10min"]})

for _, shot in pens_scored.iterrows():
    mid = shot["match_id"]
    team = shot["team"]
    minute = int(shot["game_minute"])
    tl_row = timeline[(timeline["match_id"] == mid) & (timeline["team"] == team) & (timeline["minute"] == minute)]
    if len(tl_row) > 0 and tl_row.iloc[0].get("future_window_complete_10min", 0) == 1:
        post_pen_score.append({"conceded": tl_row.iloc[0]["future_conceded_10min"],
                               "xga": tl_row.iloc[0]["future_xga_10min"]})

pm_df = pd.DataFrame(post_pen_miss) if post_pen_miss else pd.DataFrame(columns=["conceded", "xga"])
ps_df = pd.DataFrame(post_pen_score) if post_pen_score else pd.DataFrame(columns=["conceded", "xga"])

if len(pm_df) > 5 and len(ps_df) > 5:
    p_pm = pm_df["conceded"].mean() * 100
    p_ps = ps_df["conceded"].mean() * 100
    print(f"
Apres penalty RATE  : P(conceder 10min) = {p_pm:.1f}%  (n={len(pm_df)})")
    print(f"Apres penalty MARQUE: P(conceder 10min) = {p_ps:.1f}%  (n={len(ps_df)})")
    print(f"Difference : {p_pm - p_ps:+.1f} points de %")
    
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.bar([0, 1], [p_pm, p_ps], color=[RED, GREEN], edgecolor="white", linewidth=2, width=0.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Penalty rate", "Penalty marque"], fontsize=13)
    ax.set_ylabel("P(conceder dans 10 min) %")
    ax.set_title("Le penalty rate casse-t-il une equipe ?", pad=15)
    ax.text(0, p_pm + 0.5, str(round(p_pm, 1)) + "%", ha="center", fontsize=14, fontweight="bold", color=RED)
    ax.text(1, p_ps + 0.5, str(round(p_ps, 1)) + "%", ha="center", fontsize=14, fontweight="bold", color=GREEN)
    ax.text(0, -1.5, "n=" + str(len(pm_df)), ha="center", fontsize=10, color=GRAY)
    ax.text(1, -1.5, "n=" + str(len(ps_df)), ha="center", fontsize=10, color=GRAY)
    plt.tight_layout()
    plt.savefig("../outputs/figures/pub_13_penalty.png", dpi=200, bbox_inches="tight", facecolor=BG)
    plt.show()
else:
    print("Pas assez de penalties pour l analyse")

---
## 7. La mega-infographie finale

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor(BG)
fig.suptitle(""Si tu ne marques pas, tu encaisses" - Verdict final",
             fontsize=20, fontweight="bold", color=DARK, y=1.02)

# 1. Effet par fenetre
ax = axes[0, 0]
ax.set_facecolor(BG)
sub05 = wr_df[wr_df["threshold"] == 0.5]
if len(sub05) > 0:
    ax.bar(range(len(sub05)), sub05["diff"].values * 100, color=BLUE, edgecolor="white", width=0.5)
    ax.set_xticks(range(len(sub05)))
    ax.set_xticklabels([str(w) + " min" for w in sub05["window"]])
    ax.axhline(0, color=DARK, linestyle="--", linewidth=1)
ax.set_title("Aucune fenetre ne montre d effet", fontsize=13)
ax.set_ylabel("Delta P(conceder) %")

# 2. Occasion ratee
ax = axes[0, 1]
ax.set_facecolor(BG)
ax.bar([0, 1], [p_miss, p_score], color=[RED, GREEN], edgecolor="white", width=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Ratee", "Marquee"])
ax.set_title("Apres une grosse occasion", fontsize=13)
ax.set_ylabel("P(conceder) %")

# 3. Par mi-temps
ax = axes[0, 2]
ax.set_facecolor(BG)
if len(h05) >= 2:
    ax.bar([0, 1], h05["diff"].values * 100, color=[BLUE, ORANGE], edgecolor="white", width=0.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["1ere MT", "2eme MT"])
    ax.axhline(0, color=DARK, linestyle="--", linewidth=1)
ax.set_title("Meme resultat par mi-temps", fontsize=13)
ax.set_ylabel("Delta P(conceder) %")

# 4. Siege
ax = axes[1, 0]
ax.set_facecolor(BG)
if len(sg_stats) > 0:
    ax.bar(range(len(sg_stats)), sg_stats["p_concede"].values * 100,
           color=[plt.cm.YlOrRd(i/len(sg_stats)) for i in range(len(sg_stats))],
           edgecolor="white", width=0.7)
    ax.set_xticks(range(len(sg_stats)))
    ax.set_xticklabels(sg_stats["shots_bin"], fontsize=9)
    bsl = tl["future_conceded_10min"].mean() * 100
    ax.axhline(bsl, color=DARK, linestyle="--", linewidth=1)
ax.set_title("Siege : tirs sans marquer", fontsize=13)
ax.set_xlabel("Tirs sans but")
ax.set_ylabel("P(conceder) %")

# 5. Forest mini
ax = axes[1, 1]
ax.set_facecolor(BG)
ax.text(0.5, 0.5, "7/11 ligues
montrent un
effet NEGATIF

L adage est faux
PARTOUT",
        ha="center", va="center", fontsize=15, fontweight="bold", color=DARK,
        transform=ax.transAxes)
ax.set_title("A travers 11 ligues", fontsize=13)
ax.axis("off")

# 6. Verdict
ax = axes[1, 2]
ax.set_facecolor(BG)
ax.text(0.5, 0.6, "MYTHE", ha="center", va="center", fontsize=40,
        fontweight="bold", color=RED, transform=ax.transAxes)
n_s = len(shots)
n_m = shots["match_id"].nunique()
ax.text(0.5, 0.3, str(n_s) + " tirs
" + str(n_m) + " matchs
11 ligues",
        ha="center", va="center", fontsize=14, color=DARK, transform=ax.transAxes)
ax.set_title("Verdict", fontsize=13)
ax.axis("off")

plt.tight_layout()
plt.savefig("../outputs/figures/pub_14_mega_summary.png", dpi=200, bbox_inches="tight", facecolor=BG)
plt.show()

---
## Recapitulatif

Figures generees dans  :
-  — Fenetres 5/10/15 min
-  — La malediction de l occasion ratee
-  — 1ere vs 2eme mi-temps
-  — Le siege infructueux
-  — Scatter equipes (gaspillage vs concessions)
-  — Le penalty rate
-  — Infographie finale 6 panneaux